In [ ]:
from functools import partial

import numpy as np
import pandas as pd
from mxlpy import Scipy, make_protocol

from mxlmodels import Simulator, get_nguyen2026_tomato, plot


def get_idxs_of_peaks(light: pd.Series) -> list[list[int]]:
    """Return groups of indices where light equals its maximum value."""
    cnt = 0
    idx_groups = []

    arr = light.to_numpy()
    max_light = light.max()
    while cnt < len(arr):
        if arr[cnt] == max_light:
            # temporary container for all F==maxlight. For each peak it is renewed
            idxs = []
            while cnt != len(arr) and arr[cnt] == max_light:
                idxs.append(cnt)
                cnt += 1
            idx_groups.append(idxs)
        else:
            cnt += 1
    return idx_groups


def get_idxs_of_peaks_(light: pd.Series) -> list[pd.Index]:
    """Return peak indices as pandas Index objects instead of integer lists."""
    mask = light == max(light)
    return list(mask[mask].groupby((1 - mask).cumsum()).apply(lambda x: x.index).values)


def get_npq(ppfd: pd.Series, fluorescence: pd.Series) -> pd.Series:
    """Calculate non-photochemical quenching from PAM simulation peak points.

    Returns
    -------
    Fm: Fm (first element of list) and Fm' values
    NPQ: Calculated NPQ values
    tm: Exact time points of peaks in PAM trace
    Fo: Fo (first element of list) and Ft' values
    to: Exact time points of Fo and Ft' values

    """
    # container for lists. Each list contains the positions of fluorescence values for one peak
    # container for position of Fo'
    t = ppfd.index.to_numpy()
    fluo = fluorescence.to_numpy()

    # Fm is the maximal value for each peak sequence
    max_fluo_per_group = [i[np.argmax(fluo[i])] for i in get_idxs_of_peaks(ppfd)]
    index = t[max_fluo_per_group]
    Fm = fluo[max_fluo_per_group]
    return pd.Series((Fm[0] - Fm) / Fm, name="NPQ", index=index)


res = (
    Simulator(
        get_nguyen2026_tomato(),
        integrator=partial(
            Scipy,
            method="Radau",
        ),
    )
    .simulate_protocol(
        make_protocol(
            [
                (0.72, {"PPFD": 100}),
                (29.28, {"PPFD": 10}),
            ]
            * 10
        )
    )
    .get_result()
    .unwrap_or_err()
)

args = res.get_args(include_parameters=True)
npq = get_npq(args["PPFD"], args["Fluo"])


fig, (ax1, ax2) = plot.two_axes(figsize=(8, 4), grid=True)
plot.lines((npq / npq.max()), ax=ax1)
ax1.set(title="NPQ")

plot.lines((args["qL"]), ax=ax2)
ax2.set(title="qL")
plot.show()